# RAFT Pipeline on CUAD (Teacher-Student Knowledge Distillation)

A single, resumable, Kaggle-GPU (2x T4, 32GB total) notebook that builds a
Retrieval-Augmented Fine-Tuning (RAFT) dataset from the CUAD legal contracts
dataset and distills it into a small student model.

**Design notes / deviations from a generic RAFT recipe (read before running):**

- **Teacher model:** `unsloth/gemma-2-9b-it-bnb-4bit` (not the 27B variant).
  A 27B model in 4-bit is ~16GB of weights alone, which does not fit with headroom
  on a single 16GB T4, and Kaggle's 2x T4 has **no tensor parallelism** — a naive
  `device_map="auto"` split runs GPUs sequentially, layer-by-layer, so nothing is
  actually parallel and generation throughput does not improve (Unsloth's own
  model cards note 1xT4 is ~5x faster than 2xT4 for exactly this reason). The 9B
  variant fits comfortably on one T4 with room to spare, generates far faster,
  and leaves the second T4 free.
- **Unsloth is single-GPU only in the free/OSS tier.** Multi-GPU training/inference
  requires Unsloth Pro. So `FastLanguageModel` (Unsloth) is used **only** for the
  3B student in Phase 4 (which fits on one T4). The teacher (Phase 2/5) and the
  base/RAFT student during **evaluation** (Phase 5) are loaded with plain
  `transformers` + `bitsandbytes` instead, using `device_map="auto"` per the
  OOM-prevention rule, since inference doesn't require Unsloth's training kernels.
- **Ragas judge model:** rather than having the just-fine-tuned RAFT student score
  its own outputs (circular / biased), Faithfulness and Answer Relevancy are
  computed with the **teacher model reloaded as an independent judge**, wrapped
  via `LangchainLLMWrapper` per Ragas' custom-LLM integration path.
- **Rank-1/3/5 Exact Match & F1** are computed by retrieving the top-{1,3,5}
  chunks via FAISS and generating an answer conditioned on that context. The
  zero-shot config receives no retrieved context by construction, so its
  Rank-1/3/5 scores are identical (repeated) across the three columns — this is
  expected, not a bug. "Span F1 Score" and the Ragas metrics are all reported at
  k=3 as the representative retrieval depth.
- Every phase checks Hugging Face Hub for its output file first and skips
  recomputation if found (`HfApi().file_exists`). All intermediates and the final
  adapter are pushed to the Hub — nothing depends on `/kaggle/working` surviving
  a kernel restart.

**Prerequisites you must have in place before running:**
1. A Kaggle secret named `HF_TOKEN` (Add-ons → Secrets) — a Hugging Face token
   with **write** access.
2. That HF account must have accepted the Gemma license at
   `https://huggingface.co/google/gemma-2-9b-it` (gated model — ungated tokens
   will get a 401/403 on load).
3. Kaggle GPU accelerator enabled (2x T4).


In [ ]:
# !pip install -q -U --force-reinstall datasets==2.21.0
# !pip install -q protobuf>=4.25.3
# !pip install -q -U unsloth bitsandbytes "transformers>=4.44" "trl>=0.12" peft \
#     sentence-transformers faiss-cpu "ragas>=0.2,<0.3" huggingface_hub accelerate \
#     langchain langchain-huggingface langchain-community tabulate pyarrow

In [ ]:
import datasets; print(datasets.__version__)

In [ ]:
# Cell 2: Imports, Configuration, HF Login, Shared Helpers
import os, gc, sys, json, time, random, logging, traceback, string, re, collections
import numpy as np
import pandas as pd
import torch
import faiss

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", stream=sys.stdout)
logger = logging.getLogger("RAFT-Pipeline")

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Pipeline-wide parameters ----
CHUNK_SIZE_CHARS = 2000          # ~512 tokens
CHUNK_OVERLAP_CHARS = 250        # ~64 tokens
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEACHER_MODEL_NAME = "unsloth/gemma-2-9b-it-bnb-4bit"   # swapped from 27B, see notebook intro
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
STUDENT_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
N_SYNTHETIC_SAMPLES = 500
N_EVAL_SAMPLES = 50
N_DISTRACTORS = 2
GROUNDING_THRESHOLD = 0.5
BATCH_CHECKPOINT_EVERY = 50

EMBEDDED_CHUNKS_FILE = "cuad_embedded_chunks.parquet"
RAW_SYNTHETIC_FILE = "cuad_raw_synthetic.parquet"
FILTERED_RAFT_FILE = "cuad_filtered_raft.parquet"
EVAL_HOLDOUT_FILE = "cuad_eval_holdout.parquet"
EVAL_RESULTS_FILE = "evaluation_results.csv"

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)

from huggingface_hub import HfApi, login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read HF_TOKEN from Kaggle Secrets. Add a secret named 'HF_TOKEN' "
        "under Add-ons > Secrets with a Hugging Face WRITE token, and make sure that "
        "account has accepted the Gemma license at "
        "https://huggingface.co/google/gemma-2-9b-it before running this notebook."
    ) from e

login(token=hf_token)
api = HfApi()
HF_USER = api.whoami(token=hf_token)["name"]
HF_REPO_ID = f"{HF_USER}/cuad-raft-pipeline"
MODEL_REPO_ID = f"{HF_USER}/qwen2.5-3b-cuad-raft"

api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
api.create_repo(repo_id=MODEL_REPO_ID, repo_type="model", exist_ok=True)

logger.info(f"HF user: {HF_USER}")
logger.info(f"Dataset checkpoint repo: {HF_REPO_ID}")
logger.info(f"Model output repo: {MODEL_REPO_ID}")


def is_phase_complete(filename: str, repo_id: str = HF_REPO_ID, repo_type: str = "dataset") -> bool:
    '''Mandatory resume check: query HF Hub before doing any computation.'''
    try:
        return api.file_exists(repo_id=repo_id, filename=filename, repo_type=repo_type)
    except Exception as e:
        logger.warning(f"file_exists check failed for {filename}: {e}")
        return False


def push_parquet(df: pd.DataFrame, filename: str):
    local_path = os.path.join(WORKDIR, filename)
    df.to_parquet(local_path, index=False)
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message=f"Add/update {filename}",
    )
    logger.info(f"Pushed {filename} ({len(df)} rows) to {HF_REPO_ID}")


def pull_parquet(filename: str) -> pd.DataFrame:
    local_path = hf_hub_download(repo_id=HF_REPO_ID, filename=filename, repo_type="dataset")
    df = pd.read_parquet(local_path)
    logger.info(f"Pulled {filename} ({len(df)} rows) from {HF_REPO_ID}")
    return df


def clear_cuda_cache_and_log():
    '''Call AFTER del <objects> in the calling cell's own scope -- del must
    happen in the cell itself so the real references are dropped, not just a
    helper function's local parameter names.'''
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            logger.info(f"  GPU {i}: {free / 1e9:.2f} GB free / {total / 1e9:.2f} GB total")
    logger.info("Memory cleanup complete.")


def normalize_answer(s: str) -> str:
    s = (s or "").lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

logger.info("Config, HF login, and shared helpers ready.")


## PHASE 1: Dataset Preparation & Embedding Indexing

Loads `theatticusproject/cuad-qa`, deduplicates contract passages, performs
sliding-window chunking, embeds with MiniLM, checkpoints to
`cuad_embedded_chunks.parquet`, and builds an in-RAM FAISS `IndexFlatIP` index.


In [ ]:
# Phase 1: Dataset Preparation & Embedding Indexing
try:
    logger.info("=== PHASE 1: Dataset Preparation & Embedding Indexing ===")

    if is_phase_complete(EMBEDDED_CHUNKS_FILE):
        logger.info(f"{EMBEDDED_CHUNKS_FILE} found on {HF_REPO_ID}. Skipping chunking/embedding computation.")
        chunks_df = pull_parquet(EMBEDDED_CHUNKS_FILE)
    else:
        logger.info("No checkpoint found. Loading CUAD and building chunks from scratch...")
        from datasets import load_dataset

        cuad_raw = load_dataset("theatticusproject/cuad-qa", split="train")

        # Many QA rows share the same underlying contract passage -- dedupe on context text.
        seen = set()
        unique_docs = []
        for row in cuad_raw:
            ctx = row["context"]
            if ctx not in seen:
                seen.add(ctx)
                unique_docs.append(ctx)
        logger.info(f"Extracted {len(unique_docs)} unique contract passages from CUAD.")

        def sliding_window_chunks(text, size=CHUNK_SIZE_CHARS, overlap=CHUNK_OVERLAP_CHARS):
            chunks = []
            step = max(size - overlap, 1)
            for start in range(0, len(text), step):
                chunk = text[start:start + size]
                if len(chunk.strip()) > 0:
                    chunks.append(chunk)
                if start + size >= len(text):
                    break
            return chunks

        records = []
        for doc_id, doc_text in enumerate(unique_docs):
            for chunk_text in sliding_window_chunks(doc_text):
                records.append({"source_doc_id": doc_id, "text": chunk_text})

        chunk_df_raw = pd.DataFrame(records)
        chunk_df_raw["chunk_id"] = chunk_df_raw.index
        logger.info(
            f"Generated {len(chunk_df_raw)} chunks "
            f"(size={CHUNK_SIZE_CHARS} chars, overlap={CHUNK_OVERLAP_CHARS} chars)."
        )

        from sentence_transformers import SentenceTransformer

        embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
        embeddings = embed_model.encode(
            chunk_df_raw["text"].tolist(),
            batch_size=128,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        chunk_df_raw["embedding"] = embeddings.tolist()  # store as plain python lists for parquet
        chunks_df = chunk_df_raw[["chunk_id", "source_doc_id", "text", "embedding"]].reset_index(drop=True)

        push_parquet(chunks_df, EMBEDDED_CHUNKS_FILE)

        del embed_model
        clear_cuda_cache_and_log()

    # ---- Build FAISS IndexFlatIP in RAM (always rebuilt in-memory; never persisted to disk) ----
    embedding_matrix = np.vstack(
        chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy()
    )
    faiss_index = faiss.IndexFlatIP(embedding_matrix.shape[1])
    faiss_index.add(embedding_matrix)
    logger.info(f"FAISS IndexFlatIP built with {faiss_index.ntotal} vectors (dim={embedding_matrix.shape[1]}).")

except Exception:
    logger.error("Phase 1 failed.")
    logger.error(traceback.format_exc())
    raise


## PHASE 2: Synthetic Data Generation (Teacher Model)

Samples chunks, prompts the teacher (`unsloth/gemma-2-9b-it-bnb-4bit`) to write
one legal Q/A pair per chunk, assembles RAFT entries (1 oracle chunk + 2 FAISS
distractor chunks from a *different* contract, per the spec's "lowest cosine
similarity" rule), and checkpoints every 50 samples to
`cuad_raw_synthetic.parquet`. Also carves out a disjoint 50-chunk held-out pool
reused by Phase 5.


In [ ]:
# Phase 2: Synthetic Data Generation (Teacher Model)
try:
    logger.info("=== PHASE 2: Synthetic Data Generation (Teacher Model) ===")

    # ---- Deterministic, disjoint train/eval chunk sampling (seeded -> stable across resumes) ----
    candidate_chunks = chunks_df[chunks_df["text"].str.len() > 200].reset_index(drop=True)
    total_needed = min(N_SYNTHETIC_SAMPLES + N_EVAL_SAMPLES, len(candidate_chunks))
    all_sampled = candidate_chunks.sample(n=total_needed, random_state=SEED).reset_index(drop=True)
    sample_order = all_sampled.iloc[: min(N_SYNTHETIC_SAMPLES, total_needed)].reset_index(drop=True)
    eval_sample_order = all_sampled.iloc[min(N_SYNTHETIC_SAMPLES, total_needed):].reset_index(drop=True)
    logger.info(f"Training sample pool: {len(sample_order)} chunks. Held-out eval pool: {len(eval_sample_order)} chunks.")

    TEACHER_PROMPT_TEMPLATE = (
        "You are an expert legal analyst. Read the following contract section and produce "
        "1 highly specific, professional legal question and its exact factual answer based "
        "strictly on the text.\n\nContract Excerpt:\n{oracle_chunk}\n\n"
        "Output Format:\nQUESTION: <question>\nANSWER: <exact answer>"
    )

    def generate_qa(oracle_text, model, tokenizer):
        prompt = TEACHER_PROMPT_TEMPLATE.format(oracle_chunk=oracle_text)
        messages = [{"role": "user", "content": prompt}]
        # input_ids = tokenizer.apply_chat_template(
        #     messages, add_generation_prompt=True, return_tensors="pt"
        # ).to(model.device)

        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        input_ids = inputs["input_ids"].to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=300,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        return generated

    def parse_qa(generated_text):
        question, answer = None, None
        for line in generated_text.splitlines():
            line = line.strip()
            if line.upper().startswith("QUESTION:"):
                question = line.split(":", 1)[1].strip()
            elif line.upper().startswith("ANSWER:"):
                answer = line.split(":", 1)[1].strip()
        return question, answer

    def get_distractors(oracle_row, k=N_DISTRACTORS):
        '''Per spec: distractors are the LOWEST cosine-similarity chunks from a
        different contract than the oracle (i.e. random/easy negatives, matching
        the original RAFT paper's use of unrelated distractor documents).'''
        oracle_emb = np.asarray(oracle_row["embedding"], dtype=np.float32).reshape(1, -1)
        sims, idxs = faiss_index.search(oracle_emb, faiss_index.ntotal)
        candidates = []
        for sim, idx in zip(sims[0][::-1], idxs[0][::-1]):  # ascending similarity
            cand_row = chunks_df.iloc[idx]
            if cand_row["source_doc_id"] != oracle_row["source_doc_id"]:
                candidates.append(cand_row)
            if len(candidates) >= k:
                break
        return candidates

    # ---- Resume: pull existing progress, if any ----
    if is_phase_complete(RAW_SYNTHETIC_FILE):
        existing_synth_df = pull_parquet(RAW_SYNTHETIC_FILE)
    else:
        existing_synth_df = pd.DataFrame(
            columns=["sample_idx", "question", "answer", "oracle_chunk_id", "oracle_text",
                     "distractor_chunk_ids", "distractor_texts"]
        )

    start_idx = len(existing_synth_df)

    if start_idx >= len(sample_order):
        logger.info("Phase 2 synthetic generation already complete. Skipping teacher load.")
        raw_synthetic_df = existing_synth_df
    else:
        logger.info(f"Resuming generation from sample {start_idx}/{len(sample_order)}.")
        from transformers import AutoModelForCausalLM, AutoTokenizer

        teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
        teacher_model = AutoModelForCausalLM.from_pretrained(
            TEACHER_MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        teacher_model.eval()

        new_records = []
        for i in range(start_idx, len(sample_order)):
            oracle_row = sample_order.iloc[i]
            try:
                raw_output = generate_qa(oracle_row["text"], teacher_model, teacher_tokenizer)
                question, answer = parse_qa(raw_output)
                if not question or not answer:
                    logger.warning(f"Sample {i}: could not parse QUESTION/ANSWER, skipping.")
                    continue
                distractors = get_distractors(oracle_row)
                new_records.append({
                    "sample_idx": i,
                    "question": question,
                    "answer": answer,
                    "oracle_chunk_id": int(oracle_row["chunk_id"]),
                    "oracle_text": oracle_row["text"],
                    "distractor_chunk_ids": [int(d["chunk_id"]) for d in distractors],
                    "distractor_texts": [d["text"] for d in distractors],
                })
            except Exception as sample_err:
                logger.warning(f"Sample {i} generation failed: {sample_err}")
                continue

            is_last = (i == len(sample_order) - 1)
            if (i + 1) % BATCH_CHECKPOINT_EVERY == 0 or is_last:
                if new_records:
                    combined_df = pd.concat([existing_synth_df, pd.DataFrame(new_records)], ignore_index=True)
                else:
                    combined_df = existing_synth_df
                push_parquet(combined_df, RAW_SYNTHETIC_FILE)
                existing_synth_df = combined_df
                new_records = []
                logger.info(f"Checkpoint saved at sample {i + 1}/{len(sample_order)}.")

        raw_synthetic_df = existing_synth_df

        del teacher_model, teacher_tokenizer
        clear_cuda_cache_and_log()

    logger.info(f"Phase 2 complete: {len(raw_synthetic_df)} raw synthetic RAFT samples available.")

except Exception:
    logger.error("Phase 2 failed.")
    logger.error(traceback.format_exc())
    raise

## PHASE 3: Automated Grounding Filter (Cross-Encoder)

Scores each `(oracle_chunk, synthetic_answer)` pair with `BAAI/bge-reranker-base`
and drops anything below the 0.5 grounding threshold, to keep hallucinated
answers out of the fine-tuning set.


In [ ]:
# Phase 3: Automated Grounding Filter (Cross-Encoder)
try:
    logger.info("=== PHASE 3: Automated Grounding Filter ===")

    if is_phase_complete(FILTERED_RAFT_FILE):
        logger.info(f"{FILTERED_RAFT_FILE} found on {HF_REPO_ID}. Skipping grounding filter.")
        filtered_df = pull_parquet(FILTERED_RAFT_FILE)
    else:
        raw_df = pull_parquet(RAW_SYNTHETIC_FILE) if is_phase_complete(RAW_SYNTHETIC_FILE) else raw_synthetic_df.copy()

        from sentence_transformers import CrossEncoder

        reranker = CrossEncoder(RERANKER_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
        pairs = list(zip(raw_df["oracle_text"].tolist(), raw_df["answer"].tolist()))
        raw_scores = reranker.predict(pairs, show_progress_bar=True)

        # bge-reranker-base outputs an unbounded relevance logit; squash to [0,1]
        # with a sigmoid so the 0.5 threshold from the spec is meaningful.
        grounding_scores = 1.0 / (1.0 + np.exp(-np.asarray(raw_scores, dtype=np.float64)))
        raw_df = raw_df.copy()
        raw_df["grounding_score"] = grounding_scores

        filtered_df = raw_df[raw_df["grounding_score"] >= GROUNDING_THRESHOLD].reset_index(drop=True)
        logger.info(
            f"Grounding filter kept {len(filtered_df)}/{len(raw_df)} samples "
            f"(threshold={GROUNDING_THRESHOLD})."
        )

        push_parquet(filtered_df, FILTERED_RAFT_FILE)

        del reranker
        clear_cuda_cache_and_log()

    logger.info(f"Phase 3 complete: {len(filtered_df)} grounded RAFT samples ready for fine-tuning.")

except Exception:
    logger.error("Phase 3 failed.")
    logger.error(traceback.format_exc())
    raise


## PHASE 4: Student Model Fine-Tuning (Unsloth QLoRA)

Fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` on the grounded RAFT dataset
with Unsloth (single-GPU, free-tier compatible) and pushes the LoRA adapter to
the Hub.


In [ ]:
# Phase 4: Student Model Fine-Tuning (Unsloth QLoRA)
try:
    logger.info("=== PHASE 4: Student Model Fine-Tuning (Unsloth QLoRA) ===")

    adapter_already_pushed = is_phase_complete("adapter_config.json", repo_id=MODEL_REPO_ID, repo_type="model")

    if adapter_already_pushed:
        logger.info(f"adapter_config.json already present at {MODEL_REPO_ID}. Skipping fine-tuning.")
    else:
        filtered_df = pull_parquet(FILTERED_RAFT_FILE) if is_phase_complete(FILTERED_RAFT_FILE) else filtered_df.copy()
        filtered_df = filtered_df.reset_index(drop=True)

        import unsloth  # must be imported before transformers/peft/trl for Unsloth's patches to apply
        from unsloth import FastLanguageModel
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset

        student_model, student_tokenizer = FastLanguageModel.from_pretrained(
            model_name=STUDENT_MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )
        student_model = FastLanguageModel.get_peft_model(
            student_model,
            r=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=32,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
        )

        def build_chatml_example(row):
            context_chunks = [row["oracle_text"]] + list(row["distractor_texts"])
            rng = random.Random(int(row["sample_idx"]))
            rng.shuffle(context_chunks)
            context_block = "\n\n---\n\n".join(context_chunks)
            user_content = f"Context:\n{context_block}\n\nQuestion: {row['question']}"
            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
            return student_tokenizer.apply_chat_template(messages, tokenize=False)

        filtered_df["text"] = filtered_df.apply(build_chatml_example, axis=1)
        train_dataset = Dataset.from_pandas(filtered_df[["text"]])

        sft_config = SFTConfig(
            output_dir=os.path.join(WORKDIR, "qlora_outputs"),
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,   # effective batch size = 16
            num_train_epochs=2,
            learning_rate=2e-4,
            warmup_ratio=0.05,
            fp16=True,
            bf16=False,
            optim="adamw_8bit",
            logging_steps=10,
            save_strategy="no",
            seed=SEED,
            report_to=[],
        )

        trainer = SFTTrainer(
            model=student_model,
            train_dataset=train_dataset,
            processing_class=student_tokenizer,
            args=sft_config,
        )
        train_result = trainer.train()
        logger.info(f"Training finished. Final loss: {train_result.training_loss:.4f}")

        student_model.push_to_hub(MODEL_REPO_ID, token=hf_token)
        student_tokenizer.push_to_hub(MODEL_REPO_ID, token=hf_token)
        logger.info(f"Pushed fine-tuned LoRA adapter to {MODEL_REPO_ID}")

        del trainer, student_model, student_tokenizer
        clear_cuda_cache_and_log()

    logger.info("Phase 4 complete.")

except Exception:
    logger.error("Phase 4 failed.")
    logger.error(traceback.format_exc())
    raise


## PHASE 5: Evaluation & Comparison Table

Builds a 50-question held-out set (disjoint chunk pool from Phase 2), evaluates
Base Zero-Shot, Base+RAG, and Fine-Tuned RAFT at retrieval depths k=1/3/5, and
scores Ragas Faithfulness/Answer Relevancy using the **teacher model reloaded as
an independent judge** (not the fine-tuned student itself).


In [ ]:
# Phase 5: Evaluation & Comparison Table
try:
    logger.info("=== PHASE 5: Evaluation & Comparison Table ===")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from sentence_transformers import SentenceTransformer
    from peft import PeftModel

    def retrieve_topk(question, k, embed_model):
        q_emb = embed_model.encode([question], normalize_embeddings=True).astype(np.float32)
        _, idxs = faiss_index.search(q_emb, k)
        return [chunks_df.iloc[idx]["text"] for idx in idxs[0]]

    def generate_answer(model, tokenizer, question, context_chunks=None, max_new_tokens=150):
        if context_chunks:
            context_block = "\n\n---\n\n".join(context_chunks)
            user_content = (
                f"Context:\n{context_block}\n\nQuestion: {question}\n"
                "Answer concisely and factually based only on the context."
            )
        else:
            user_content = f"Question: {question}\nAnswer concisely and factually."
        messages = [{"role": "user", "content": user_content}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        input_ids = inputs["input_ids"].to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

    # ---- 5a. Teacher reloaded for (a) eval-set generation if needed, (b) independent Ragas judge ----
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    teacher_model.eval()
    logger.info("Teacher model reloaded (held-out set generation + Ragas judge duty).")

    if is_phase_complete(EVAL_HOLDOUT_FILE):
        eval_df = pull_parquet(EVAL_HOLDOUT_FILE)
    else:
        eval_records = []
        for i, row in eval_sample_order.iterrows():
            try:
                raw_output = generate_qa(row["text"], teacher_model, teacher_tokenizer)
                question, answer = parse_qa(raw_output)
                if not question or not answer:
                    logger.warning(f"Eval sample {i}: could not parse QUESTION/ANSWER, skipping.")
                    continue
                eval_records.append({
                    "eval_idx": int(i),
                    "question": question,
                    "gold_answer": answer,
                    "oracle_chunk_id": int(row["chunk_id"]),
                })
            except Exception as eval_err:
                logger.warning(f"Eval sample {i} generation failed: {eval_err}")
                logger.warning(traceback.format_exc())
        eval_df = pd.DataFrame(eval_records)
        push_parquet(eval_df, EVAL_HOLDOUT_FILE)

    logger.info(f"Held-out evaluation set ready: {len(eval_df)} questions.")

    # ---- 5b. Shared retrieval (model-independent) at k = 1, 3, 5 ----
    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
    for k in (1, 3, 5):
        eval_df[f"ctx_k{k}"] = eval_df["question"].apply(lambda q, kk=k: retrieve_topk(q, kk, embed_model))
    del embed_model
    clear_cuda_cache_and_log()

    # ---- 5c. Config A: Base Student, Zero-Shot ----
    base_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
    base_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    base_model.eval()

    eval_df["pred_zero_shot"] = [
        generate_answer(base_model, base_tokenizer, q, context_chunks=None) for q in eval_df["question"]
    ]
    logger.info("Base Zero-Shot predictions complete.")

    # ---- 5d. Config B: Base Student + RAG (k = 1, 3, 5) ----
    for k in (1, 3, 5):
        eval_df[f"pred_rag_k{k}"] = [
            generate_answer(base_model, base_tokenizer, q, context_chunks=ctx)
            for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
        ]
    logger.info("Base + RAG predictions complete.")

    del base_model, base_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5e. Config C: Fine-Tuned RAFT Student (k = 1, 3, 5) ----
    raft_tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO_ID)
    raft_base_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
    )
    raft_model = PeftModel.from_pretrained(raft_base_model, MODEL_REPO_ID)
    raft_model.eval()

    for k in (1, 3, 5):
        eval_df[f"pred_raft_k{k}"] = [
            generate_answer(raft_model, raft_tokenizer, q, context_chunks=ctx)
            for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
        ]
    logger.info("Fine-Tuned RAFT predictions complete.")

    del raft_model, raft_base_model, raft_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5f. Exact Match / Span F1 ----
    metrics = {
        "Base Model (Zero-Shot)": {},
        "Base Model + RAG": {},
        "Fine-Tuned RAFT Model": {},
    }

    def mean_em_f1(pred_col, gold_col="gold_answer"):
        ems = [exact_match(p, g) for p, g in zip(eval_df[pred_col], eval_df[gold_col])]
        f1s = [f1_score(p, g) for p, g in zip(eval_df[pred_col], eval_df[gold_col])]
        return float(np.mean(ems)), float(np.mean(f1s))

    rank_rows = ["Exact Match @ Rank-1", "Exact Match @ Rank-3", "Exact Match @ Rank-5"]

    # Zero-shot never sees retrieved context, so its "rank" scores are identical
    # across k -- reported once and repeated, as noted in the intro markdown cell.
    em_zs, f1_zs = mean_em_f1("pred_zero_shot")
    for row in rank_rows:
        metrics["Base Model (Zero-Shot)"][row] = em_zs
    metrics["Base Model (Zero-Shot)"]["Span F1 Score"] = f1_zs

    for label, prefix in [("Base Model + RAG", "pred_rag"), ("Fine-Tuned RAFT Model", "pred_raft")]:
        for k, row in zip((1, 3, 5), rank_rows):
            em, _ = mean_em_f1(f"{prefix}_k{k}")
            metrics[label][row] = em
        _, f1_at_3 = mean_em_f1(f"{prefix}_k3")
        metrics[label]["Span F1 Score"] = f1_at_3

    # ---- 5g. Ragas Faithfulness / Answer Relevancy (teacher as independent judge) ----
    try:
        from datasets import Dataset as HFDataset
        from transformers import pipeline as hf_pipeline
        from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
        from ragas import evaluate as ragas_evaluate
        from ragas import EvaluationDataset
        from ragas.metrics import Faithfulness, AnswerRelevancy
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper

        judge_text_gen_pipe = hf_pipeline(
            "text-generation",
            model=teacher_model,
            tokenizer=teacher_tokenizer,
            max_new_tokens=300,
            do_sample=False,
        )
        judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge_text_gen_pipe))
        judge_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME))

        def compute_ragas_scores(answer_col):
            ragas_hf_ds = HFDataset.from_dict({
                "user_input": eval_df["question"].tolist(),
                "response": eval_df[answer_col].tolist(),
                "retrieved_contexts": eval_df["ctx_k3"].tolist(),
            })
            ragas_eval_ds = EvaluationDataset.from_hf_dataset(ragas_hf_ds)
            result = ragas_evaluate(
                dataset=ragas_eval_ds,
                metrics=[Faithfulness(), AnswerRelevancy()],
                llm=judge_llm,
                embeddings=judge_embeddings,
            )
            result_df = result.to_pandas()
            return float(result_df["faithfulness"].mean()), float(result_df["answer_relevancy"].mean())

        for label, answer_col in [
            ("Base Model (Zero-Shot)", "pred_zero_shot"),
            ("Base Model + RAG", "pred_rag_k3"),
            ("Fine-Tuned RAFT Model", "pred_raft_k3"),
        ]:
            try:
                faith, rel = compute_ragas_scores(answer_col)
            except Exception as ragas_err:
                logger.warning(f"Ragas scoring failed for '{label}': {ragas_err}")
                faith, rel = float("nan"), float("nan")
            metrics[label]["Ragas Faithfulness"] = faith
            metrics[label]["Ragas Answer Relevancy"] = rel

    except Exception:
        logger.warning("Ragas evaluation block failed; filling NaN so the notebook still completes.")
        logger.warning(traceback.format_exc())
        for label in metrics:
            metrics[label].setdefault("Ragas Faithfulness", float("nan"))
            metrics[label].setdefault("Ragas Answer Relevancy", float("nan"))

    del teacher_model, teacher_tokenizer
    clear_cuda_cache_and_log()

    # ---- 5h. Assemble & print comparison table ----
    row_order = rank_rows + ["Span F1 Score", "Ragas Faithfulness", "Ragas Answer Relevancy"]
    col_order = ["Base Model (Zero-Shot)", "Base Model + RAG", "Fine-Tuned RAFT Model"]

    comparison_df = pd.DataFrame(index=row_order, columns=col_order, dtype=float)
    for col in col_order:
        for row in row_order:
            comparison_df.loc[row, col] = metrics[col][row]
    comparison_df.index.name = "Evaluation Metric"

    print(comparison_df.reset_index().to_markdown(index=False, floatfmt=".4f"))

    local_csv_path = os.path.join(WORKDIR, EVAL_RESULTS_FILE)
    comparison_df.reset_index().to_csv(local_csv_path, index=False)
    api.upload_file(
        path_or_fileobj=local_csv_path,
        path_in_repo=EVAL_RESULTS_FILE,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message="Add evaluation_results.csv",
    )
    logger.info(f"Pushed {EVAL_RESULTS_FILE} to {HF_REPO_ID}")
    logger.info("Phase 5 complete. RAFT pipeline finished end-to-end.")

except Exception:
    logger.error("Phase 5 failed.")
    logger.error(traceback.format_exc())
    raise


In [ ]:
!pip install -q langchain-google-vertexai

In [ ]:
# # === Ragas for RAFT k3 Only ===
# import sys
# from unittest.mock import MagicMock
# sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
# sys.modules['langchain_community.chat_models'] = MagicMock()
# sys.modules['langchain_community.llms'] = MagicMock()

# from transformers import pipeline as hf_pipeline
# from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
# from ragas import evaluate as ragas_evaluate
# from ragas import EvaluationDataset
# from ragas.metrics import Faithfulness, AnswerRelevancy
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from datasets import Dataset as HFDataset

# import os, gc, numpy as np, pandas as pd, torch, logging, traceback
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
# logger = logging.getLogger("RAFT-Ragas")

# from huggingface_hub import hf_hub_download
# HF_REPO_ID = "abhifdsdf/cuad-raft-pipeline"

# local_path = hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset")
# eval_df = pd.read_parquet(local_path)
# chunks_path = hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset")
# chunks_df = pd.read_parquet(chunks_path)
# import faiss
# embedding_matrix = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
# faiss_index = faiss.IndexFlatIP(embedding_matrix.shape[1])
# faiss_index.add(embedding_matrix)

# from sentence_transformers import SentenceTransformer
# embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
# def retrieve_topk(q, k):
#     q_emb = embed_model.encode([q], normalize_embeddings=True).astype(np.float32)
#     _, idxs = faiss_index.search(q_emb, k)
#     return [chunks_df.iloc[idx]["text"] for idx in idxs[0]]
# for k in (1, 3, 5):
#     eval_df[f"ctx_k{k}"] = eval_df["question"].apply(lambda q, kk=k: retrieve_topk(q, kk))
# del embed_model; gc.collect()

# # Generate RAFT predictions
# from unsloth import FastLanguageModel
# from peft import PeftModel

# raft_base, raft_tokenizer = FastLanguageModel.from_pretrained(
#     "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True,
# )
# raft_model = PeftModel.from_pretrained(raft_base, "abhifdsdf/qwen2.5-3b-cuad-raft")
# raft_model.eval()

# def gen_answer(model, tokenizer, question, ctx=None):
#     if ctx:
#         ctx_block = "\n\n---\n\n".join(ctx)
#         prompt = f"<|im_start|>user\nContext:\n{ctx_block}\n\nQuestion: {question}\nAnswer concisely and factually based only on the context.<|im_end|>\n<|im_start|>assistant\n"
#     else:
#         prompt = f"<|im_start|>user\nQuestion: {question}\nAnswer concisely and factually.<|im_end|>\n<|im_start|>assistant\n"
#     inputs = tokenizer(prompt, return_tensors="pt")
#     input_ids = inputs["input_ids"].to(model.device)
#     with torch.no_grad():
#         out = model.generate(input_ids, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
#     return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

# eval_df["pred_raft_k3"] = [gen_answer(raft_model, raft_tokenizer, q, ctx) for q, ctx in zip(eval_df["question"], eval_df["ctx_k3"])]
# del raft_model, raft_base, raft_tokenizer; gc.collect(); torch.cuda.empty_cache()

# # Load teacher via Unsloth for Ragas
# teacher_model, teacher_tokenizer = FastLanguageModel.from_pretrained(
#     "unsloth/gemma-2-9b-it-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True,
# )
# teacher_model.eval()

# from transformers import pipeline as hf_pipeline
# from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
# from ragas import evaluate as ragas_evaluate
# from ragas import EvaluationDataset
# from ragas.metrics import Faithfulness, AnswerRelevancy
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from datasets import Dataset as HFDataset

# judge_pipe = hf_pipeline("text-generation", model=teacher_model, tokenizer=teacher_tokenizer, max_new_tokens=300, do_sample=False)
# judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge_pipe))
# judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

# hf_ds = HFDataset.from_dict({
#     "user_input": eval_df["question"].tolist(),
#     "response": eval_df["pred_raft_k3"].tolist(),
#     "retrieved_contexts": eval_df["ctx_k3"].tolist(),
# })
# import os; os.environ["TOKENIZERS_PARALLELISM"] = "false"
# from ragas.run_config import RunConfig

# faith = Faithfulness()
# relev = AnswerRelevancy()

# result = ragas_evaluate(
#     dataset=EvaluationDataset.from_hf_dataset(hf_ds),
#     metrics=[faith, relev],
#     llm=judge_llm,
#     embeddings=judge_emb,
#     run_config=RunConfig(max_workers=1, max_retries=0),
# )
# rdf = result.to_pandas()
# print(f"Faithfulness: {rdf['faithfulness'].mean():.4f}")
# print(f"AnswerRelevancy: {rdf['answer_relevancy'].mean():.4f}")

# run form next

In [ ]:
# # === Ragas on 10 samples (9B judge, fits 16GB) ===
# import sys; from unittest.mock import MagicMock
# sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
# sys.modules['langchain_community.chat_models'] = MagicMock()
# sys.modules['langchain_community.llms'] = MagicMock()
# import os; os.environ["TOKENIZERS_PARALLELISM"] = "false"
# import gc, numpy as np, pandas as pd, torch, logging
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
# logger = logging.getLogger("RAFT-Ragas")

# from huggingface_hub import hf_hub_download
# HF_REPO_ID = "abhifdsdf/cuad-raft-pipeline"

# eval_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset"))
# eval_df = eval_df.head(10).reset_index(drop=True)  # ONLY 10 SAMPLES
# logger.info(f"Using {len(eval_df)} samples for Ragas")

# chunks_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset"))
# import faiss
# em = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
# idx = faiss.IndexFlatIP(em.shape[1]); idx.add(em)

# from sentence_transformers import SentenceTransformer
# embed = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
# def rt(q, k):
#     qe = embed.encode([q], normalize_embeddings=True).astype(np.float32)
#     _, ids = idx.search(qe, k)
#     return [chunks_df.iloc[i]["text"] for i in ids[0]]
# eval_df["ctx_k3"] = eval_df["question"].apply(lambda q: rt(q, 3))
# del embed; gc.collect()

# # Load student, generate 10 predictions, delete
# from unsloth import FastLanguageModel
# from peft import PeftModel
# student, stok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
# student_model = PeftModel.from_pretrained(student, "abhifdsdf/qwen2.5-3b-cuad-raft")
# student_model.eval()
# def gen(q, ctx):
#     c = "\n\n---\n\n".join(ctx)
#     p = f"<|im_start|>user\nContext:\n{c}\n\nQuestion: {q}\nAnswer concisely and factually based only on the context.<|im_end|>\n<|im_start|>assistant\n"
#     inp = stok(p, return_tensors="pt")["input_ids"].to(student_model.device)
#     with torch.no_grad():
#         out = student_model.generate(inp, max_new_tokens=150, do_sample=False, pad_token_id=stok.eos_token_id)
#     return stok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True).strip()
# eval_df["pred_raft_k3"] = [gen(q, ctx) for q, ctx in zip(eval_df["question"], eval_df["ctx_k3"])]
# del student, student_model, stok; gc.collect(); torch.cuda.empty_cache()

# # Load 9B teacher (only model on GPU now)
# teacher, tok = FastLanguageModel.from_pretrained("unsloth/gemma-2-9b-it-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
# teacher.eval()

# from transformers import pipeline as hf_pipeline
# from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
# from ragas import evaluate as ragas_evaluate, EvaluationDataset
# from ragas.metrics import Faithfulness, AnswerRelevancy
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from ragas.run_config import RunConfig
# from datasets import Dataset as HFDataset

# pipe = hf_pipeline("text-generation", model=teacher, tokenizer=tok, max_new_tokens=300, do_sample=False)
# llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=pipe))
# ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

# ds = HFDataset.from_dict({
#     "user_input": eval_df["question"].tolist(),
#     "response": eval_df["pred_raft_k3"].tolist(),
#     "retrieved_contexts": eval_df["ctx_k3"].tolist(),
# })
# result = ragas_evaluate(
#     dataset=EvaluationDataset.from_hf_dataset(ds),
#     metrics=[Faithfulness(), AnswerRelevancy()],
#     llm=llm,
#     embeddings=ragas_emb,
#     run_config=RunConfig(max_workers=1, max_retries=0),
# )
# rdf = result.to_pandas()
# print(f"\n=== RAGAS on 10 samples (9B judge) ===")
# print(f"Faithfulness:     {rdf['faithfulness'].mean():.4f}")
# print(f"AnswerRelevancy:  {rdf['answer_relevancy'].mean():.4f}")

In [ ]:
# import os
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import gc, numpy as np, pandas as pd, torch, logging
import faiss
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from peft import PeftModel

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("RAFT-Gen")

HF_REPO_ID = "abhifdsdf/cuad-raft-pipeline"

# 1. Load Data & Embeddings
eval_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset"))
chunks_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset"))

em = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
idx = faiss.IndexFlatIP(em.shape[1])
idx.add(em)

# 2. Retrieval
embed = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
def rt(q, k):
    qe = embed.encode([q], normalize_embeddings=True).astype(np.float32)
    _, ids = idx.search(qe, k)
    return [chunks_df.iloc[i]["text"] for i in ids[0]]

logger.info("Running retrieval...")
eval_df["ctx_k3"] = eval_df["question"].apply(lambda q: rt(q, 3))
del embed
gc.collect(); torch.cuda.empty_cache()

# 3. Load Student Model
logger.info("Loading student model...")
student, stok = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True,
)
student_model = PeftModel.from_pretrained(student, "abhifdsdf/qwen2.5-3b-cuad-raft")
student_model.eval()

def gen(q, ctx=None):
    c = "\n\n---\n\n".join(ctx) if ctx else ""
    p = f"<|im_start|>user\nContext:\n{c}\n\nQuestion: {q}\nAnswer concisely and factually based only on the context.<|im_end|>\n<|im_start|>assistant\n" if ctx else f"<|im_start|>user\nQuestion: {q}\nAnswer concisely and factually.<|im_end|>\n<|im_start|>assistant\n"
    inp = stok(p, return_tensors="pt")["input_ids"].to(student_model.device)
    with torch.no_grad():
        out = student_model.generate(inp, max_new_tokens=150, do_sample=False, pad_token_id=stok.eos_token_id)
    return stok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True).strip()

logger.info("Generating predictions...")
eval_df["pred_raft_k3"] = [gen(q, ctx) for q, ctx in zip(eval_df["question"], eval_df["ctx_k3"])]

# 4. Save and Exit
eval_df.to_parquet("eval_with_preds.parquet")
logger.info("Done. Predictions saved to eval_with_preds.parquet")

In [ ]:
print("here")

In [ ]:
# === Ragas with ChatTemplateLLM ===
from typing import Optional, List, Any
import sys; from unittest.mock import MagicMock
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.llms'] = MagicMock()
import os; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import gc, numpy as np, pandas as pd, torch, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("RAFT-Ragas")
from tqdm import tqdm

from huggingface_hub import hf_hub_download
HF_REPO_ID = "abhifdsdf/cuad-raft-pipeline"
N_SAMPLES = 15

eval_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset"))
eval_df = eval_df.head(N_SAMPLES).reset_index(drop=True)
logger.info(f"Using {len(eval_df)} samples")

chunks_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset"))
import faiss
em = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
idx = faiss.IndexFlatIP(em.shape[1]); idx.add(em)
from sentence_transformers import SentenceTransformer
embed = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
def rt(q, k):
    qe = embed.encode([q], normalize_embeddings=True).astype(np.float32)
    _, ids = idx.search(qe, k)
    return [chunks_df.iloc[i]["text"] for i in ids[0]]
eval_df["ctx_k3"] = eval_df["question"].apply(lambda q: rt(q, 3))
del embed; gc.collect(); torch.cuda.empty_cache()

from unsloth import FastLanguageModel
from peft import PeftModel
student, stok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
student = PeftModel.from_pretrained(student, "abhifdsdf/qwen2.5-3b-cuad-raft")
student.eval()
eval_df["pred_raft_k3"] = [(
    lambda q, ctx: stok.decode(
        student.generate(
            input_ids := stok.apply_chat_template(
                [{"role":"user","content":f"Context:\n{chr(10).join(ctx)}\n\nQuestion: {q}\nAnswer concisely and factually based only on the context."}],
                tokenize=True, add_generation_prompt=True, return_tensors="pt"
            ).to("cuda"),
            max_new_tokens=150, do_sample=False, pad_token_id=stok.eos_token_id,
            attention_mask=torch.ones_like(input_ids)
        )[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()
)(eval_df.iloc[i]["question"], eval_df.iloc[i]["ctx_k3"]) for i in tqdm(range(len(eval_df)), desc="Student gen")]
del student; gc.collect(); torch.cuda.empty_cache()

from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

class ChatTemplateLLM(LLM):
    model: Any
    tokenizer: Any
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs) -> str:
        msgs = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(msgs, tokenize=False)
        inp = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inp, max_new_tokens=512, do_sample=False, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    
    @property
    def _llm_type(self) -> str: return "chat_template_qwen"

judge, jtok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-7B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
judge.eval()
judge_llm = ChatTemplateLLM(model=judge, tokenizer=jtok)

from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate as ragas_evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from datasets import Dataset as HFDataset

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

ds = HFDataset.from_dict({
    "user_input": eval_df["question"].tolist(),
    "response": eval_df["pred_raft_k3"].tolist(),
    "retrieved_contexts": eval_df["ctx_k3"].tolist(),
})
result = ragas_evaluate(
    dataset=EvaluationDataset.from_hf_dataset(ds),
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=RunConfig(max_workers=1, max_retries=0),
)
rdf = result.to_pandas()
print(f"\n=== RAGAS ({N_SAMPLES}s, ChatTemplateLLM) ===")
print(f"Faithfulness:     {rdf['faithfulness'].mean():.4f}")
print(f"AnswerRelevancy:  {rdf['answer_relevancy'].mean():.4f}")

In [1]:
print("here1")

here1


In [2]:
import sys; from unittest.mock import MagicMock
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.llms'] = MagicMock()
import os; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import gc, numpy as np, pandas as pd, torch, logging, json
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("RAFT-Ragas")
from tqdm import tqdm

from huggingface_hub import hf_hub_download
HF_REPO_ID = "abhifdsdf/cuad-raft-pipeline"
N_SAMPLES = 15

eval_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset"))
eval_df = eval_df.head(N_SAMPLES).reset_index(drop=True)

chunks_df = pd.read_parquet(hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset"))
import faiss
em = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
idx = faiss.IndexFlatIP(em.shape[1]); idx.add(em)
from sentence_transformers import SentenceTransformer
embed = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
def rt(q, k):
    qe = embed.encode([q], normalize_embeddings=True).astype(np.float32)
    _, ids = idx.search(qe, k)
    return [chunks_df.iloc[i]["text"] for i in ids[0]]
eval_df["ctx_k3"] = eval_df["question"].apply(lambda q: rt(q, 3))
del embed; gc.collect(); torch.cuda.empty_cache()

# Student gen
from unsloth import FastLanguageModel
from peft import PeftModel
student, stok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
student = PeftModel.from_pretrained(student, "abhifdsdf/qwen2.5-3b-cuad-raft")
student.eval()
preds = []
for i in tqdm(range(len(eval_df)), desc="Student"):
    q, ctx = eval_df.iloc[i]["question"], eval_df.iloc[i]["ctx_k3"]
    msgs = [{"role": "user", "content": f"Context:\n{chr(10).join(ctx)}\n\nQuestion: {q}\nAnswer concisely and factually based only on the context."}]
    inp = stok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = student.generate(inp, max_new_tokens=150, do_sample=False, pad_token_id=stok.eos_token_id, attention_mask=torch.ones_like(inp))
    preds.append(stok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True).strip())
eval_df["pred"] = preds
del student; gc.collect(); torch.cuda.empty_cache()

# Judge
judge, jtok = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-7B-Instruct-bnb-4bit", max_seq_length=2048, dtype=None, load_in_4bit=True)
judge.eval()

def gen(msgs):
    inp = jtok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = judge.generate(inp, max_new_tokens=512, do_sample=False, pad_token_id=jtok.eos_token_id, attention_mask=torch.ones_like(inp))
    return jtok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True).strip()

# === Faithfulness ===
def extract_claims(ans, q, ctx):
    resp = gen([{"role": "user", "content": f"Extract each individual factual claim from the answer below. Output a JSON array of strings, one claim per element.\n\nQuestion: {q}\nAnswer: {ans}\n\n[\"claim 1\", \"claim 2\"]"}])
    try: return json.loads(resp)
    except: return [s.strip("- ").strip() for s in resp.split("\n") if s.strip() and len(s.strip()) > 5]

def verify(c, ctx, q):
    resp = gen([{"role": "user", "content": f"Context:\n{ctx[:2000]}\n\nClaim: {c}\nQuestion: {q}\nIs this claim TRUE or FALSE based only on the context? Answer TRUE or FALSE."}])
    return 1.0 if resp.strip().upper().startswith("TRUE") else 0.0

# === Answer Relevancy ===
from sentence_transformers import SentenceTransformer
sem = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

faith, rel = [], []
for i in tqdm(range(len(eval_df)), desc="Ragas"):
    r = eval_df.iloc[i]; ctx = "\n\n---\n\n".join(r["ctx_k3"])
    
    claims = extract_claims(r["pred"], r["question"], ctx)
    fs = [verify(c, ctx, r["question"]) for c in claims] if claims else [0.0]
    faith.append(np.mean(fs))
    
    resp = gen([{"role": "user", "content": f"Generate 3 questions that this answer answers. Output a JSON array.\n\nAnswer: {r['pred']}\n\n["}])
    try: qs = json.loads(resp)
    except: qs = [r["pred"]]
    if qs:
        ea = sem.encode([r["question"]] + qs)
        rel.append(np.mean([float(np.dot(ea[0], e)/(np.linalg.norm(ea[0])*np.linalg.norm(e))) for e in ea[1:]]))
    else: rel.append(0.0)

print(f"\n=== RAGAS ({N_SAMPLES}s) ===")
print(f"Faithfulness:     {np.mean(faith):.4f}")
print(f"AnswerRelevancy:  {np.mean(rel):.4f}")

2026-07-26 10:23:40,779 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_eval_holdout.parquet "HTTP/1.1 302 Found"
2026-07-26 10:23:40,880 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_embedded_chunks.parquet "HTTP/1.1 302 Found"
2026-07-26 10:23:41,220 | INFO | Loading faiss with AVX512 support.
2026-07-26 10:23:41,221 | INFO | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-07-26 10:23:41,222 | INFO | Loading faiss with AVX2 support.
2026-07-26 10:23:41,223 | INFO | Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-07-26 10:23:41,223 | INFO | Loading faiss.
2026-07-26 10:23:41,258 | INFO | Successfully loaded faiss.
2026-07-26 10:23:47,932 | WARNING | Skipping import of cpp extensions due to incompatible torch version. Please upgra

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-26 10:23:54,932 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:23:55,001 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:23:55,074 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:23:55,145 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.js

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Ragas:  93%|█████████▎| 14/15 [04:34<00:19, 19.13s/it]Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingfa

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Ragas: 100%|██████████| 15/15 [05:12<00:00, 20.84s/it]


=== RAGAS (15s) ===
Faithfulness:     0.3467
AnswerRelevancy:  0.5428


In [ ]:
# # === Ragas Fix + Semantic Similarity Metrics ===
# !pip install -q langchain-google-vertexai  # fixes ragas import

# import os, gc, json, traceback, logging, numpy as np, pandas as pd, torch
# logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
# logger = logging.getLogger("RAFT-Eval")

# from huggingface_hub import HfApi, hf_hub_download, login
# from kaggle_secrets import UserSecretsClient
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")
# login(token=hf_token)
# api = HfApi()
# HF_USER = api.whoami(token=hf_token)["name"]
# HF_REPO_ID = f"{HF_USER}/cuad-raft-pipeline"
# MODEL_REPO_ID = f"{HF_USER}/qwen2.5-3b-cuad-raft"

# # ---- Load eval_df (questions + gold answers) ----
# local_path = hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_eval_holdout.parquet", repo_type="dataset")
# eval_df = pd.read_parquet(local_path)
# logger.info(f"Loaded eval holdout: {len(eval_df)} questions")

# # ---- Load chunks + FAISS (needed for retrieval + RAG) ----
# chunks_path = hf_hub_download(repo_id=HF_REPO_ID, filename="cuad_embedded_chunks.parquet", repo_type="dataset")
# chunks_df = pd.read_parquet(chunks_path)
# import faiss
# embedding_matrix = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
# faiss_index = faiss.IndexFlatIP(embedding_matrix.shape[1])
# faiss_index.add(embedding_matrix)

# def retrieve_topk(question, k, embed_model):
#     q_emb = embed_model.encode([question], normalize_embeddings=True).astype(np.float32)
#     _, idxs = faiss_index.search(q_emb, k)
#     return [chunks_df.iloc[idx]["text"] for idx in idxs[0]]

# # ---- Load embed model ----
# from sentence_transformers import SentenceTransformer
# embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
# for k in (1, 3, 5):
#     eval_df[f"ctx_k{k}"] = eval_df["question"].apply(lambda q, kk=k: retrieve_topk(q, kk, embed_model))
# del embed_model
# gc.collect()

# # ---- Helper: generate answer ----
# from transformers import AutoModelForCausalLM, AutoTokenizer

# def generate_answer(model, tokenizer, question, context_chunks=None, max_new_tokens=150):
#     if context_chunks:
#         context_block = "\n\n---\n\n".join(context_chunks)
#         user_content = (
#             f"Context:\n{context_block}\n\nQuestion: {question}\n"
#             "Answer concisely and factually based only on the context."
#         )
#     else:
#         user_content = f"Question: {question}\nAnswer concisely and factually."
#     prompt = f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"
#     inputs = tokenizer(prompt, return_tensors="pt")
#     input_ids = inputs["input_ids"].to(model.device)
#     with torch.no_grad():
#         out_ids = model.generate(
#             input_ids, max_new_tokens=max_new_tokens, do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     return tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()


# # ---- Load base model ----
# from peft import PeftModel
# logger.info("Loading base model...")
# base_tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit")
# base_model = AutoModelForCausalLM.from_pretrained(
#     "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", device_map="auto", torch_dtype=torch.float16
# )
# base_model.eval()

# # ---- Generate zero-shot + RAG predictions ----
# eval_df["pred_zero_shot"] = [generate_answer(base_model, base_tokenizer, q) for q in eval_df["question"]]
# logger.info("Zero-shot done.")
# for k in (1, 3, 5):
#     eval_df[f"pred_rag_k{k}"] = [
#         generate_answer(base_model, base_tokenizer, q, context_chunks=ctx)
#         for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
#     ]
# logger.info("Base + RAG done.")
# del base_model, base_tokenizer
# gc.collect(); torch.cuda.empty_cache()

# # ---- Load RAFT model ----
# logger.info("Loading RAFT model...")
# raft_tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-3B-Instruct-bnb-4bit")
# raft_base = AutoModelForCausalLM.from_pretrained(
#     "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", device_map="auto", torch_dtype=torch.float16
# )
# raft_model = PeftModel.from_pretrained(raft_base, MODEL_REPO_ID)
# raft_model.eval()

# for k in (1, 3, 5):
#     eval_df[f"pred_raft_k{k}"] = [
#         generate_answer(raft_model, raft_tokenizer, q, context_chunks=ctx)
#         for q, ctx in zip(eval_df["question"], eval_df[f"ctx_k{k}"])
#     ]
# logger.info("RAFT predictions done.")
# del raft_model, raft_base, raft_tokenizer
# gc.collect(); torch.cuda.empty_cache()

# # ---- Semantic similarity metric (cosine) ----
# logger.info("Computing semantic similarity...")
# from sentence_transformers import SentenceTransformer
# sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# def semantic_sim(pred_col, gold_col="gold_answer"):
#     pred_embs = sim_model.encode(eval_df[pred_col].tolist(), normalize_embeddings=True)
#     gold_embs = sim_model.encode(eval_df[gold_col].tolist(), normalize_embeddings=True)
#     return float(np.mean(np.sum(pred_embs * gold_embs, axis=1)))

# for label, pred_col in [
#     ("Base Zero-Shot", "pred_zero_shot"),
#     ("Base + RAG", "pred_rag_k3"),
#     ("Fine-Tuned RAFT", "pred_raft_k3"),
# ]:
#     sim = semantic_sim(pred_col)
#     em = np.mean([int(a.strip().lower() == b.strip().lower()) for a, b in zip(eval_df[pred_col], eval_df["gold_answer"])])
#     logger.info(f"{label:25s} | Semantic Sim: {sim:.4f} | Exact Match: {em:.4f}")

# del sim_model
# gc.collect()
# torch.cuda.empty_cache()

# # ---- Ragas ----
# logger.info("Loading teacher model as Ragas judge...")
# teacher_tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-2-9b-it-bnb-4bit")
# teacher_model = AutoModelForCausalLM.from_pretrained(
#     "unsloth/gemma-2-9b-it-bnb-4bit", device_map="auto", torch_dtype=torch.float16,
# )
# teacher_model.eval()

# from transformers import pipeline as hf_pipeline
# from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
# from ragas import evaluate as ragas_evaluate
# from ragas import EvaluationDataset
# from ragas.metrics import Faithfulness, AnswerRelevancy
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from datasets import Dataset as HFDataset

# judge_pipe = hf_pipeline("text-generation", model=teacher_model, tokenizer=teacher_tokenizer, max_new_tokens=300, do_sample=False)
# judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge_pipe))
# judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

# for label, answer_col in [
#     ("Base Zero-Shot", "pred_zero_shot"),
#     ("Base + RAG", "pred_rag_k3"),
#     ("Fine-Tuned RAFT", "pred_raft_k3"),
# ]:
#     try:
#         hf_ds = HFDataset.from_dict({
#             "user_input": eval_df["question"].tolist(),
#             "response": eval_df[answer_col].tolist(),
#             "retrieved_contexts": eval_df["ctx_k3"].tolist(),
#         })
#         result = ragas_evaluate(
#             dataset=EvaluationDataset.from_hf_dataset(hf_ds),
#             metrics=[Faithfulness(), AnswerRelevancy()],
#             llm=judge_llm,
#             embeddings=judge_emb,
#         )
#         rdf = result.to_pandas()
#         faith = float(rdf["faithfulness"].mean())
#         relev = float(rdf["answer_relevancy"].mean())
#         logger.info(f"{label:25s} | Faithfulness: {faith:.4f} | AnswerRelevancy: {relev:.4f}")
#     except Exception as e:
#         logger.warning(f"{label} Ragas failed: {e}")
#         logger.warning(traceback.format_exc())

# del teacher_model, teacher_tokenizer
# gc.collect(); torch.cuda.empty_cache()
# logger.info("All metrics complete.")

In [ ]:
# # === EXPORT CELL: Merge + Convert to GGUF + Push ===
# import os, torch, gc, subprocess, shutil
# from huggingface_hub import HfApi
# from unsloth import FastLanguageModel

# MERGE_DIR = "/kaggle/working/merged_model"
# GGUF_DIR = "/kaggle/working/gguf_output"
# os.makedirs(MERGE_DIR, exist_ok=True)
# os.makedirs(GGUF_DIR, exist_ok=True)

# api = HfApi()

# # 1. Merge LoRA into base model and save as fp16 safetensors
# logger.info("Loading student model + LoRA adapter for merge...")
# student_model, student_tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_REPO_ID,
#     max_seq_length=MAX_SEQ_LENGTH,
#     dtype=None,
#     load_in_4bit=True,
# )
# student_model = FastLanguageModel.get_peft_model(
#     student_model,
#     r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha=32,
#     lora_dropout=0,
#     bias="none",
#     use_gradient_checkpointing="unsloth",
#     random_state=SEED,
# )

# logger.info(f"Merging LoRA and saving as fp16 to {MERGE_DIR}...")
# student_model.save_pretrained_merged(
#     MERGE_DIR,
#     student_tokenizer,
#     save_method="merged_16bit",
# )
# logger.info("Merged fp16 model saved.")

# # 2. Push merged model to HF Hub
# GGUF_REPO = f"{HF_USER}/qwen2.5-3b-cuad-raft-gguf"
# api.create_repo(repo_id=GGUF_REPO, repo_type="model", exist_ok=True)

# logger.info(f"Pushing merged model to {GGUF_REPO}...")
# student_model.push_to_hub_merged(
#     GGUF_REPO,
#     student_tokenizer,
#     save_method="merged_16bit",
#     token=hf_token,
# )
# logger.info("Merged model pushed to Hub.")

# del student_model
# gc.collect()
# torch.cuda.empty_cache()

# # 3. Install llama.cpp and convert to GGUF
# logger.info("Installing llama.cpp for GGUF conversion...")
# os.chdir("/kaggle/working")
# if not os.path.exists("llama.cpp"):
#     subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggerganov/llama.cpp"], check=True)

# logger.info("Converting to GGUF Q4_K_M...")
# subprocess.run([
#     "python", "llama.cpp/convert_hf_to_gguf.py",
#     MERGE_DIR,
#     "--outfile", f"{GGUF_DIR}/qwen-cuad-raft-q4_k_m.gguf",
#     "--outtype", "q4_k_m",
# ], check=True)
# logger.info("GGUF conversion complete.")

# gguf_path = f"{GGUF_DIR}/qwen-cuad-raft-q4_k_m.gguf"
# gguf_size = os.path.getsize(gguf_path) / 1e9
# logger.info(f"GGUF file size: {gguf_size:.2f} GB")

# # 4. Upload GGUF to Hub
# logger.info(f"Uploading GGUF to {GGUF_REPO}...")
# api.upload_file(
#     path_or_fileobj=gguf_path,
#     path_in_repo="qwen-cuad-raft-q4_k_m.gguf",
#     repo_id=GGUF_REPO,
#     repo_type="model",
#     commit_message="Add GGUF Q4_K_M",
# )
# logger.info(f"GGUF uploaded to {GGUF_REPO}")
# logger.info("Export complete!")

In [ ]:
# # === Corrected GGUF Conversion ===
# import os, subprocess, shutil
# from huggingface_hub import HfApi

# GGUF_REPO = f"{HF_USER}/qwen2.5-3b-cuad-raft-gguf"
# GGUF_DIR = "/kaggle/working/gguf_output"
# os.makedirs(GGUF_DIR, exist_ok=True)

# # The merged model was saved to /kaggle/working/abhifdsdf/qwen2.5-3b-cuad-raft-gguf
# # by Unsloth's push_to_hub_merged. It's already in the working dir.
# MERGE_DIR = f"/kaggle/working/merged_model"

# # Step 1: Convert HF model to FP16 GGUF
# # logger.info("Converting HF model to FP16 GGUF...")
# # subprocess.run([
# #     "python", "llama.cpp/convert_hf_to_gguf.py",
# #     MERGE_DIR,
# #     "--outfile", f"{GGUF_DIR}/qwen-cuad-raft-f16.gguf",
# #     "--outtype", "f16",
# # ], check=True)
# # logger.info("FP16 GGUF created.")

# # Step 2: Build llama-quantize if needed
# quantize_bin = "llama.cpp/build/bin/Release/llama-quantize"
# if not os.path.exists(quantize_bin):
#     logger.info("Building llama-quantize...")
#     os.chdir("llama.cpp")
#     subprocess.run(["cmake", "-B", "build"], check=True, capture_output=True)
#     subprocess.run(["cmake", "--build", "build", "--config", "Release", "--target", "llama-quantize"], check=True, capture_output=True)
#     os.chdir("/kaggle/working")
#     quantize_bin = "llama.cpp/build/bin/llama-quantize"

# # Step 3: Quantize to Q4_K_M
# logger.info("Quantizing to Q4_K_M...")
# subprocess.run([
#     quantize_bin,
#     f"{GGUF_DIR}/qwen-cuad-raft-f16.gguf",
#     f"{GGUF_DIR}/qwen-cuad-raft-q4_k_m.gguf",
#     "q4_k_m",
# ], check=True)

# gguf_size = os.path.getsize(f"{GGUF_DIR}/qwen-cuad-raft-q4_k_m.gguf") / 1e9
# logger.info(f"Q4_K_M GGUF created: {gguf_size:.2f} GB")

# # Step 4: Upload to HF
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=f"{GGUF_DIR}/qwen-cuad-raft-q4_k_m.gguf",
#     path_in_repo="qwen-cuad-raft-q4_k_m.gguf",
#     repo_id=GGUF_REPO,
#     repo_type="model",
#     commit_message="Add GGUF Q4_K_M",
# )
# logger.info(f"GGUF uploaded to {GGUF_REPO}")

# # Cleanup intermediate file
# os.remove(f"{GGUF_DIR}/qwen-cuad-raft-f16.gguf")
# logger.info("Done!")

In [ ]:
import subprocess, glob
result = subprocess.run(["cmake", "--build", "llama.cpp/build", "--config", "Release", "--target", "llama-quantize"], capture_output=True, text=True)
print(result.stdout[-500:])
print(result.stderr[-500:])
# Find the binary
found = list(glob.glob("llama.cpp/build/**/llama-quantize*", recursive=True))
print("Found binaries:", found)

In [3]:
# Cell 1: Config + HF login (reuses the same repos as the main pipeline)
import os, gc, sys, json, random, logging, traceback, string, re, collections, zipfile
import numpy as np
import pandas as pd
import torch
import faiss
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", stream=sys.stdout)
logger = logging.getLogger("Showcase-Assets")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
STUDENT_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

EMBEDDED_CHUNKS_FILE = "cuad_embedded_chunks.parquet"
RAW_SYNTHETIC_FILE = "cuad_raw_synthetic.parquet"
FILTERED_RAFT_FILE = "cuad_filtered_raft.parquet"
EVAL_HOLDOUT_FILE = "cuad_eval_holdout.parquet"
EVAL_RESULTS_FILE = "evaluation_results.csv"

ASSETS_DIR = "/kaggle/working/website_assets"
GRAPHS_DIR = os.path.join(ASSETS_DIR, "graphs")
DATA_DIR = os.path.join(ASSETS_DIR, "data")
os.makedirs(GRAPHS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

from huggingface_hub import HfApi, login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not read HF_TOKEN from Kaggle Secrets (Add-ons > Secrets).") from e

login(token=hf_token)
api = HfApi()
HF_USER = api.whoami(token=hf_token)["name"]
HF_REPO_ID = f"{HF_USER}/cuad-raft-pipeline"
MODEL_REPO_ID = f"{HF_USER}/qwen2.5-3b-cuad-raft"
ASSETS_REPO_ID = f"{HF_USER}/cuad-raft-showcase-assets"

api.create_repo(repo_id=ASSETS_REPO_ID, repo_type="dataset", exist_ok=True)
logger.info(f"Source pipeline repo: {HF_REPO_ID} | Model repo: {MODEL_REPO_ID} | Assets repo: {ASSETS_REPO_ID}")

# ---- Site palette, shared with the Next.js theme, so charts drop straight into the dark UI ----
INK = "#10131A"
PAPER = "#F7F4EC"
REDLINE = "#B33A2E"
GOLD = "#E8B93F"
TEAL = "#3FA7A0"
TEXT_LIGHT = "#EDEAE2"
GRID = "#2A2E37"

plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "savefig.facecolor": "none",
    "axes.edgecolor": GRID,
    "axes.labelcolor": TEXT_LIGHT,
    "xtick.color": TEXT_LIGHT,
    "ytick.color": TEXT_LIGHT,
    "text.color": TEXT_LIGHT,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "font.size": 12,
    "font.family": "DejaVu Sans",
})


def savefig(fig, name):
    path = os.path.join(GRAPHS_DIR, name)
    fig.savefig(path, dpi=170, bbox_inches="tight", transparent=True)
    plt.close(fig)
    logger.info(f"Saved {path}")


def pull_parquet_safe(filename, repo_id=HF_REPO_ID):
    try:
        if not api.file_exists(repo_id=repo_id, filename=filename, repo_type="dataset"):
            logger.warning(f"{filename} not found on {repo_id}.")
            return None
        local_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")
        return pd.read_parquet(local_path)
    except Exception as e:
        logger.warning(f"Could not pull {filename}: {e}")
        return None


def normalize_answer(s):
    s = (s or "").lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = collections.Counter(pt) & collections.Counter(gt)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision, recall = num_same / len(pt), num_same / len(gt)
    return 2 * precision * recall / (precision + recall)

logger.info("Config ready.")


2026-07-26 10:30:49,205 | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-26 10:30:49,279 | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-07-26 10:30:49,353 | INFO | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
2026-07-26 10:30:49,355 | INFO | Source pipeline repo: abhifdsdf/cuad-raft-pipeline | Model repo: abhifdsdf/qwen2.5-3b-cuad-raft | Assets repo: abhifdsdf/cuad-raft-showcase-assets
2026-07-26 10:30:49,357 | INFO | Config ready.


In [4]:
# Cell 2: Pull pipeline artifacts back down from the Hub
chunks_df = pull_parquet_safe(EMBEDDED_CHUNKS_FILE)
raw_synthetic_df = pull_parquet_safe(RAW_SYNTHETIC_FILE)
filtered_df = pull_parquet_safe(FILTERED_RAFT_FILE)
eval_df = pull_parquet_safe(EVAL_HOLDOUT_FILE)

assert chunks_df is not None, "cuad_embedded_chunks.parquet missing -- run Phase 1 of the main pipeline first."
assert raw_synthetic_df is not None, "cuad_raw_synthetic.parquet missing -- run Phase 2 first."
assert filtered_df is not None, "cuad_filtered_raft.parquet missing -- run Phase 3 first."
assert eval_df is not None, "cuad_eval_holdout.parquet missing -- run Phase 5 (at least the eval-set-generation half) first."

logger.info(f"chunks_df: {len(chunks_df)} rows | raw_synthetic_df: {len(raw_synthetic_df)} rows | "
            f"filtered_df: {len(filtered_df)} rows | eval_df: {len(eval_df)} rows")

# Try to pull the aggregate metrics table if Phase 5 finished and pushed it; otherwise we
# recompute EM/F1 aggregates fresh below (Ragas scores can only come from the original run,
# since recomputing them here would need the teacher model reloaded again).
existing_metrics_df = None
if api.file_exists(repo_id=HF_REPO_ID, filename=EVAL_RESULTS_FILE, repo_type="dataset"):
    local_path = hf_hub_download(repo_id=HF_REPO_ID, filename=EVAL_RESULTS_FILE, repo_type="dataset")
    existing_metrics_df = pd.read_csv(local_path)
    logger.info("Found an existing evaluation_results.csv from Phase 5 -- will use its Ragas scores.")
else:
    logger.warning("No evaluation_results.csv on the Hub yet -- Ragas Faithfulness/Relevancy will be "
                    "reported as null in metrics.json. EM/F1 will still be computed fresh below.")

faiss_index = faiss.IndexFlatIP(np.vstack(chunks_df["embedding"].apply(np.asarray).to_numpy()).shape[1])
faiss_index.add(np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy()))
logger.info(f"FAISS index rebuilt: {faiss_index.ntotal} vectors.")


2026-07-26 10:30:52,667 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_embedded_chunks.parquet "HTTP/1.1 302 Found"
2026-07-26 10:30:52,747 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_embedded_chunks.parquet "HTTP/1.1 302 Found"
2026-07-26 10:30:53,140 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_raw_synthetic.parquet "HTTP/1.1 302 Found"
2026-07-26 10:30:53,218 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_raw_synthetic.parquet "HTTP/1.1 302 Found"
2026-07-26 10:30:53,321 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/cuad_filtered_raft.parquet "HTTP/1.1 302 Found"
2026-07-26 10:30:53,395 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/abhifdsdf/cuad-raft-pipeline/resolve/main/

In [5]:
# Cell 3: Dataset / pipeline stat graphs (no GPU needed)

# --- Funnel: unique passages -> chunks -> synthetic samples -> kept after grounding filter ---
funnel_labels = ["Unique\ncontract passages", "Chunks\nindexed", "Synthetic Q/A\ngenerated", "Kept after\ngrounding filter"]
funnel_values = [
    int(chunks_df["source_doc_id"].nunique()),
    len(chunks_df),
    len(raw_synthetic_df),
    len(filtered_df),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(funnel_labels, funnel_values, color=[TEAL, TEAL, GOLD, REDLINE], width=0.6)
for b, v in zip(bars, funnel_values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom", fontsize=11, color=TEXT_LIGHT)
ax.set_ylabel("Count")
ax.set_title("RAFT dataset construction funnel")
savefig(fig, "dataset_funnel.png")

# --- Chunk length distribution ---
chunk_lengths = chunks_df["text"].str.len()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(chunk_lengths, bins=30, color=TEAL, alpha=0.85)
ax.axvline(chunk_lengths.mean(), color=GOLD, linestyle="--", linewidth=1.5, label=f"mean = {chunk_lengths.mean():.0f} chars")
ax.set_xlabel("Chunk length (characters)")
ax.set_ylabel("Count")
ax.set_title("Chunk length distribution")
ax.legend()
savefig(fig, "chunk_length_distribution.png")

# --- Grounding score distribution ---
if "grounding_score" in raw_synthetic_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(raw_synthetic_df["grounding_score"], bins=30, color=GOLD, alpha=0.85)
    ax.axvline(0.5, color=REDLINE, linestyle="--", linewidth=1.5, label="grounding threshold = 0.5")
    ax.set_xlabel("Cross-encoder grounding score")
    ax.set_ylabel("Count")
    ax.set_title("Grounding score distribution (Phase 3 filter)")
    ax.legend()
    savefig(fig, "grounding_score_distribution.png")
else:
    logger.warning("raw_synthetic_df has no grounding_score column (Phase 3 may not have run against this file). Skipping.")

# --- 2D PCA projection of chunk embeddings ---
from sklearn.decomposition import PCA
emb_matrix = np.vstack(chunks_df["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)).to_numpy())
proj = PCA(n_components=2, random_state=SEED).fit_transform(emb_matrix)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(proj[:, 0], proj[:, 1], s=6, alpha=0.5, color=TEAL, linewidths=0)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Contract chunk embedding space ({len(chunks_df)} chunks, PCA of MiniLM 384-d)")
savefig(fig, "embedding_pca.png")

print("Dataset/pipeline graphs done.")


2026-07-26 10:30:55,659 | INFO | Saved /kaggle/working/website_assets/graphs/dataset_funnel.png
2026-07-26 10:30:55,894 | INFO | Saved /kaggle/working/website_assets/graphs/chunk_length_distribution.png
2026-07-26 10:30:55,896 | WARNING | raw_synthetic_df has no grounding_score column (Phase 3 may not have run against this file). Skipping.
2026-07-26 10:30:56,315 | INFO | Saved /kaggle/working/website_assets/graphs/embedding_pca.png


Dataset/pipeline graphs done.


In [7]:
# Cell 4: Regenerate per-example predictions for the held-out eval set
# (the main pipeline only ever persisted the AGGREGATE metrics table, not per-example
# predictions -- this cell regenerates them so the comparison page has real examples).
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from peft import PeftModel

def retrieve_topk(question, k, embed_model):
    q_emb = embed_model.encode([question], normalize_embeddings=True).astype(np.float32)
    _, idxs = faiss_index.search(q_emb, k)
    return [chunks_df.iloc[idx]["text"] for idx in idxs[0]]


def generate_answer(model, tokenizer, question, context_chunks=None, max_new_tokens=150):
    if context_chunks:
        context_block = "\n\n---\n\n".join(context_chunks)
        user_content = (
            f"Context:\n{context_block}\n\nQuestion: {question}\n"
            "Answer concisely and factually based only on the context."
        )
    else:
        user_content = f"Question: {question}\nAnswer concisely and factually."
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device) if isinstance(inputs, dict) else inputs.to(model.device)
    with torch.no_grad():
        out_ids = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()


N_COMPARISON_EXAMPLES = min(30, len(eval_df))  # keep the showcase page snappy; raise if you want more
comparison_df = eval_df.sample(n=N_COMPARISON_EXAMPLES, random_state=SEED).reset_index(drop=True)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
comparison_df["ctx_k3"] = comparison_df["question"].apply(lambda q: retrieve_topk(q, 3, embed_model))
del embed_model
gc.collect(); torch.cuda.empty_cache()

# Base model (zero-shot + RAG)
base, tok = FastLanguageModel.from_pretrained(STUDENT_MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True)
comparison_df["pred_zero_shot"] = [generate_answer(base, tok, q) for q in comparison_df["question"]]
comparison_df["pred_rag"] = [generate_answer(base, tok, q, ctx) for q, ctx in zip(comparison_df["question"], comparison_df["ctx_k3"])]
del base; gc.collect(); torch.cuda.empty_cache()

# RAFT model
raft, _ = FastLanguageModel.from_pretrained(STUDENT_MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True)
raft = PeftModel.from_pretrained(raft, MODEL_REPO_ID)
comparison_df["pred_raft"] = [generate_answer(raft, tok, q, ctx) for q, ctx in zip(comparison_df["question"], comparison_df["ctx_k3"])]
del raft; gc.collect(); torch.cuda.empty_cache()
del tok; gc.collect(); torch.cuda.empty_cache()
logger.info("RAFT predictions regenerated.")

for col in ["pred_zero_shot", "pred_rag", "pred_raft"]:
    comparison_df[f"em_{col}"] = [exact_match(p, g) for p, g in zip(comparison_df[col], comparison_df["gold_answer"])]
    comparison_df[f"f1_{col}"] = [f1_score(p, g) for p, g in zip(comparison_df[col], comparison_df["gold_answer"])]

print(comparison_df[["question", "pred_zero_shot", "pred_rag", "pred_raft",
                      "f1_pred_zero_shot", "f1_pred_rag", "f1_pred_raft"]].head(10))


2026-07-26 10:35:20,133 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:35:20,149 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-26 10:35:20,226 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:35:20,242 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-26 10:35:20,244 | INFO | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-07-26 10:35:20,311 | INFO | HTTP Request: HEAD https://huggingface.co/s

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-26 10:35:20,999 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:35:21,069 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:35:21,138 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-26 10:35:21,208 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.js

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


2026-07-26 10:39:56,678 | INFO | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
2026-07-26 10:39:56,780 | INFO | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/tree/b632e7c464e861a6f1762dd396048ab1ed7a10ec?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-26 10:39:56,864 | INFO | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:39:56,883 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
2026-07-26 10:39:56,904 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
2026-07-26 10:39:56,955 | INFO | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

2026-07-26 10:40:03,264 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-3B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:40:03,281 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-3B-Instruct-bnb-4bit/2672b588848527e456d320c11926c794539f47d5/generation_config.json "HTTP/1.1 200 OK"
2026-07-26 10:40:03,355 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-3B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:40:03,372 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-3B-Instruct-bnb-4bit/2672b588848527e456d320c11926c794539f47d5/config.json "HTTP/1.1 200 OK"
2026-07-26 10:40:03,463 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-3B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:40:03,481 | INFO | HTTP R

                                            question  \
0   What constitutes a "Recall" under this contract?   
1  Is the initial franchise fee refundable under ...   
2  What specific actions must the Customer take i...   
3  What is the primary responsibility of the JDC ...   
4  Does the contract require HOC to indemnify the...   
5  What are the minimum net worth requirements fo...   
6  What specific types of Intellectual Property R...   
7  What is the maximum amount of liability a part...   
8  What is the timeframe for Party A to refund th...   
9  What specific assets are conveyed to Certegy I...   

                                      pred_zero_shot  \
0  A "Recall" under the contract typically refers...   
1  The refundability of an initial franchise fee ...   
2  If a third party makes a claim against the Con...   
3  The primary responsibility of the JDC chair, a...   
4  To accurately answer this question, I would ne...   
5  The specific minimum net worth requirements 

In [8]:
# Cell 5: EM/F1 + Ragas bar charts, and comparison_examples.json / metrics.json / pipeline_stats.json

# ---- EM by rank + Span F1 (from existing_metrics_df if Phase 5 finished, else from comparison_df) ----
rank_rows = ["Exact Match @ Rank-1", "Exact Match @ Rank-3", "Exact Match @ Rank-5"]
col_order = ["Base Model (Zero-Shot)", "Base Model + RAG", "Fine-Tuned RAFT Model"]

if existing_metrics_df is not None:
    m = existing_metrics_df.set_index(existing_metrics_df.columns[0])
    em_values = {c: [m.loc[r, c] for r in rank_rows] for c in col_order}
    f1_values = {c: m.loc["Span F1 Score", c] for c in col_order}
    ragas_faith = {c: m.loc["Ragas Faithfulness", c] if "Ragas Faithfulness" in m.index else None for c in col_order}
    ragas_rel = {c: m.loc["Ragas Answer Relevancy", c] if "Ragas Answer Relevancy" in m.index else None for c in col_order}
else:
    # Fresh EM/F1 from the 30-example comparison run (single "rank" -- k=3 only, no Ragas available)
    em_zero, f1_zero = comparison_df["em_pred_zero_shot"].mean(), comparison_df["f1_pred_zero_shot"].mean()
    em_rag, f1_rag = comparison_df["em_pred_rag"].mean(), comparison_df["f1_pred_rag"].mean()
    em_raft, f1_raft = comparison_df["em_pred_raft"].mean(), comparison_df["f1_pred_raft"].mean()
    em_values = {
        "Base Model (Zero-Shot)": [em_zero] * 3,
        "Base Model + RAG": [em_rag] * 3,
        "Fine-Tuned RAFT Model": [em_raft] * 3,
    }
    f1_values = {"Base Model (Zero-Shot)": f1_zero, "Base Model + RAG": f1_rag, "Fine-Tuned RAFT Model": f1_raft}
    ragas_faith = {c: None for c in col_order}
    ragas_rel = {c: None for c in col_order}
    logger.warning("Using freshly computed k=3-only EM/F1 (single value repeated across ranks) "
                    "since the full Rank-1/3/5 sweep only exists if Phase 5 finished on the Hub.")

# --- EM by rank grouped bar chart ---
x = np.arange(len(rank_rows))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
for i, (label, color) in enumerate(zip(col_order, [TEAL, GOLD, REDLINE])):
    ax.bar(x + (i - 1) * width, em_values[label], width, label=label, color=color)
ax.set_xticks(x)
ax.set_xticklabels(["Rank-1", "Rank-3", "Rank-5"])
ax.set_ylabel("Exact Match")
ax.set_title("Exact Match by retrieval depth")
ax.legend()
savefig(fig, "em_by_rank.png")

# --- Span F1 bar chart ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(col_order, [f1_values[c] for c in col_order], color=[TEAL, GOLD, REDLINE])
ax.set_ylabel("Span F1")
ax.set_title("Span F1 Score (k=3)")
ax.set_xticklabels(col_order, rotation=12, ha="right")
savefig(fig, "f1_comparison.png")

# --- Ragas chart (only if we have real values) ---
if all(v is not None for v in ragas_faith.values()):
    x = np.arange(len(col_order))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - 0.2, [ragas_faith[c] for c in col_order], 0.4, label="Faithfulness", color=TEAL)
    ax.bar(x + 0.2, [ragas_rel[c] for c in col_order], 0.4, label="Answer Relevancy", color=GOLD)
    ax.set_xticks(x)
    ax.set_xticklabels(col_order, rotation=12, ha="right")
    ax.set_title("Ragas judge scores (teacher model as independent judge)")
    ax.legend()
    savefig(fig, "ragas_scores.png")
else:
    logger.warning("No Ragas scores available (Phase 5 evaluation_results.csv not found) -- skipping ragas_scores.png.")

# ---- pipeline_stats.json ----
pipeline_stats = {
    "unique_contract_passages": int(chunks_df["source_doc_id"].nunique()),
    "total_chunks_indexed": int(len(chunks_df)),
    "avg_chunk_length_chars": float(chunk_lengths.mean()),
    "synthetic_samples_generated": int(len(raw_synthetic_df)),
    "samples_kept_after_grounding_filter": int(len(filtered_df)),
    "grounding_retention_rate": float(len(filtered_df) / max(len(raw_synthetic_df), 1)),
    "held_out_eval_questions": int(len(eval_df)),
    "teacher_model": "unsloth/gemma-2-9b-it-bnb-4bit",
    "student_model": STUDENT_MODEL_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": "BAAI/bge-reranker-base",
    "lora_config": {"r": 16, "lora_alpha": 32, "lora_dropout": 0,
                     "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]},
}
with open(os.path.join(DATA_DIR, "pipeline_stats.json"), "w") as f:
    json.dump(pipeline_stats, f, indent=2)

# ---- metrics.json ----
metrics_json = {
    "columns": col_order,
    "rows": [
        {"metric": row, **{c: em_values[c][i] for c in col_order}}
        for i, row in enumerate(rank_rows)
    ] + [
        {"metric": "Span F1 Score", **{c: f1_values[c] for c in col_order}},
        {"metric": "Ragas Faithfulness", **{c: ragas_faith[c] for c in col_order}},
        {"metric": "Ragas Answer Relevancy", **{c: ragas_rel[c] for c in col_order}},
    ],
    "has_ragas": all(v is not None for v in ragas_faith.values()),
}
with open(os.path.join(DATA_DIR, "metrics.json"), "w") as f:
    json.dump(metrics_json, f, indent=2)

# ---- comparison_examples.json ----
examples = []
for _, row in comparison_df.iterrows():
    examples.append({
        "question": row["question"],
        "context": row["ctx_k3"],
        "gold_answer": row["gold_answer"],
        "predictions": {
            "zero_shot": {"answer": row["pred_zero_shot"], "em": int(row["em_pred_zero_shot"]), "f1": round(float(row["f1_pred_zero_shot"]), 3)},
            "rag": {"answer": row["pred_rag"], "em": int(row["em_pred_rag"]), "f1": round(float(row["f1_pred_rag"]), 3)},
            "raft": {"answer": row["pred_raft"], "em": int(row["em_pred_raft"]), "f1": round(float(row["f1_pred_raft"]), 3)},
        },
    })

raft_beats_both = sum(
    1 for e in examples
    if e["predictions"]["raft"]["f1"] > e["predictions"]["zero_shot"]["f1"]
    and e["predictions"]["raft"]["f1"] >= e["predictions"]["rag"]["f1"]
)
summary = {
    "n_examples": len(examples),
    "avg_f1": {
        "zero_shot": round(float(comparison_df["f1_pred_zero_shot"].mean()), 3),
        "rag": round(float(comparison_df["f1_pred_rag"].mean()), 3),
        "raft": round(float(comparison_df["f1_pred_raft"].mean()), 3),
    },
    "raft_best_or_tied_rate": round(raft_beats_both / len(examples), 3) if examples else None,
}
examples_sorted = sorted(examples, key=lambda e: e["predictions"]["raft"]["f1"] - e["predictions"]["zero_shot"]["f1"], reverse=True)

with open(os.path.join(DATA_DIR, "comparison_examples.json"), "w") as f:
    json.dump({"summary": summary, "examples": examples_sorted}, f, indent=2)

logger.info(f"Summary: {summary}")
print("All JSON assets written to", DATA_DIR)


2026-07-26 11:02:16,101 | INFO | Saved /kaggle/working/website_assets/graphs/em_by_rank.png
/tmp/ipykernel_10538/4089880017.py:47: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(col_order, rotation=12, ha="right")
2026-07-26 11:02:16,275 | INFO | Saved /kaggle/working/website_assets/graphs/f1_comparison.png
2026-07-26 11:02:16,469 | INFO | Saved /kaggle/working/website_assets/graphs/ragas_scores.png
2026-07-26 11:02:16,481 | INFO | Summary: {'n_examples': 30, 'avg_f1': {'zero_shot': 0.191, 'rag': 0.328, 'raft': 0.423}, 'raft_best_or_tied_rate': 0.533}


All JSON assets written to /kaggle/working/website_assets/data


In [9]:
# Cell 6: Zip everything and push to the Hub
zip_path = "/kaggle/working/website_assets.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(ASSETS_DIR):
        for fname in files:
            full_path = os.path.join(root, fname)
            arcname = os.path.relpath(full_path, ASSETS_DIR)
            zf.write(full_path, arcname)
logger.info(f"Zipped to {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")

# Push individual files (not just the zip) so they're browsable/fetchable directly from the Hub too
for root, _, files in os.walk(ASSETS_DIR):
    for fname in files:
        full_path = os.path.join(root, fname)
        arcname = os.path.relpath(full_path, ASSETS_DIR)
        api.upload_file(
            path_or_fileobj=full_path,
            path_in_repo=arcname,
            repo_id=ASSETS_REPO_ID,
            repo_type="dataset",
            commit_message=f"Add {arcname}",
        )
api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo="website_assets.zip",
    repo_id=ASSETS_REPO_ID,
    repo_type="dataset",
    commit_message="Add website_assets.zip",
)

logger.info(f"All assets pushed to https://huggingface.co/datasets/{ASSETS_REPO_ID}")
print(f"DONE. Download from Kaggle's Output panel (website_assets.zip) or from "
      f"https://huggingface.co/datasets/{ASSETS_REPO_ID}/tree/main")


2026-07-26 11:02:16,540 | INFO | Zipped to /kaggle/working/website_assets.zip (0.59 MB)
2026-07-26 11:02:16,703 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-07-26 11:02:18,567 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/commit/main "HTTP/1.1 200 OK"
2026-07-26 11:02:18,659 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

2026-07-26 11:02:24,791 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/commit/main "HTTP/1.1 200 OK"
2026-07-26 11:02:24,894 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/preupload/main "HTTP/1.1 200 OK"
2026-07-26 11:02:25,199 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/commit/main "HTTP/1.1 200 OK"
2026-07-26 11:02:25,287 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/preupload/main "HTTP/1.1 200 OK"
2026-07-26 11:02:25,675 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/commit/main "HTTP/1.1 200 OK"
2026-07-26 11:02:25,864 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/preupload/main "HTTP/1.1 200 OK"
2026-07-26 11:02:26,321 | INFO | HTTP Request: POST https://hugging

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-07-26 11:02:27,791 | INFO | HTTP Request: POST https://huggingface.co/api/datasets/abhifdsdf/cuad-raft-showcase-assets/commit/main "HTTP/1.1 200 OK"
2026-07-26 11:02:27,792 | INFO | All assets pushed to https://huggingface.co/datasets/abhifdsdf/cuad-raft-showcase-assets


DONE. Download from Kaggle's Output panel (website_assets.zip) or from https://huggingface.co/datasets/abhifdsdf/cuad-raft-showcase-assets/tree/main
